# Lab 1.5 &mdash; Challenge &mdash; The Decision Rubric, Made Executable

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Encode the &ldquo;do I need multiple agents?&rdquo; rubric as code that terminates early
- Write the acceptance bar down <i>before</i> you look at any result
- Feed Lab 1.4's real pass rates in and let them overrule the architecture you wanted
- Have an agent produce the design-review record as a typed object, not a paragraph

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The take-home artifact.** This is the one thing from Module 1 you will use next
> week: a rubric that answers the question before anyone starts building.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

Most multi-agent systems exist because the diagram was appealing, not because a question was
asked. The rubric below asks four questions in a fixed order and **stops at the first one that
decides**. Order matters: a cheaper architecture that answers the requirement wins, and asking
about elegance before asking about need is how teams talk themselves into a supervisor.

Then the measurement gets a veto. A rubric that cannot be overruled by evidence is just a
preference with a flowchart.

## Section 1 &mdash; The rubric, as code

Four questions, asked in order, first decisive answer wins:

1. Is the work **deterministic**? &rarr; a workflow, no agent at all.
2. Does it fit **one context** with a handful of tools? &rarr; a single agent.
3. Do the parts need **separate authority** &mdash; different credentials, independent audit,
   independent deployment? &rarr; supervisor and workers.
4. Otherwise &rarr; peers that negotiate. Rare, and expensive.

In [ ]:
from pydantic import BaseModel, Field

VERDICTS = ("workflow", "single_agent", "supervisor_worker", "peer_to_peer")

def rubric(deterministic: bool, fits_one_agent: bool, separate_authority: bool) -> str:
    """Return the first verdict the answers decide. Order is the design."""
    if deterministic:
        return "workflow"
    if fits_one_agent:
        return "single_agent"         # cheapest thing that works, before any question of elegance
    if separate_authority:
        return "supervisor_worker"
    return "peer_to_peer"

In [ ]:
# --- Self-check: Section 1
check("deterministic work needs no agent",
      lambda: rubric(True, False, True) == "workflow")
check("determinism is asked first",
      lambda: rubric(True, True, True) == "workflow",
      "if it can be a workflow, nothing later in the rubric should be able to override that")
check("work that fits one context gets one agent",
      lambda: rubric(False, True, False) == "single_agent")
check("fitting one context beats wanting separate authority",
      lambda: rubric(False, True, True) == "single_agent",
      "asking about authority first is how a team talks itself into a supervisor")
check("separate authority earns a supervisor",
      lambda: rubric(False, False, True) == "supervisor_worker")
check("peer-to-peer is the residue, never the goal",
      lambda: rubric(False, False, False) == "peer_to_peer")
check("every path returns a known verdict",
      lambda: all(rubric(a, b, c) in VERDICTS
                  for a in (0, 1) for b in (0, 1) for c in (0, 1)))

## Section 2 &mdash; The acceptance bar, written down first

Write the acceptance bar **before** you see a result, or you will fit it to whatever you got.
This is the same discipline as pre-registering an experiment, and it is the only reason the veto
in Section 3 means anything.

Note what is on the bar and what is not. **Silent failures** are there because Lab 1.4 showed you
what they look like: a fluent answer assembled from a lookup that quietly returned nothing. Cost
appears once, as latency, because a caller who will not wait is a real constraint. Token price is
not on the bar at all &mdash; at the scale most teams run, it is the least of the things that will
go wrong.

In [ ]:
BAR = {
    "min_pass_rate":      0.80,       # below this the architecture is not a candidate at all
    "max_silent_failures": 0,         # a wrong answer that reports no error -- Lab 1.4's bug
    "max_seconds_case":   30.0,       # the one cost clause: an answer nobody waits for is no answer
    "min_quality_gain":   0.10,       # a split must buy at least this much pass rate to be worth it
}

def meets_bar(m: dict) -> tuple[bool, str]:
    """m: {"pass_rate", "silent_failures", "seconds_case"}. Return (ok, first failing reason)."""
    if m["pass_rate"] < BAR["min_pass_rate"]:
        return False, f"pass rate {m['pass_rate']:.0%} below {BAR['min_pass_rate']:.0%}"
    if m["silent_failures"] > BAR["max_silent_failures"]:
        return False, (f"{m['silent_failures']} answer(s) wrong with nothing in the trace saying so")
    if m["seconds_case"] > BAR["max_seconds_case"]:
        return False, f"{m['seconds_case']:.1f}s/case over {BAR['max_seconds_case']}"
    return True, "meets the bar"

In [ ]:
# --- Self-check: Section 2
_ok     = {"pass_rate": 0.9,  "silent_failures": 0, "seconds_case": 8.0}
_slow   = {"pass_rate": 0.9,  "silent_failures": 0, "seconds_case": 99.0}
_silent = {"pass_rate": 0.9,  "silent_failures": 2, "seconds_case": 8.0}
_bad    = {"pass_rate": 0.40, "silent_failures": 0, "seconds_case": 1.0}

check("a good result passes",                 lambda: meets_bar(_ok)[0] is True)
check("a slow one is rejected",               lambda: meets_bar(_slow)[0] is False,
      "an answer nobody waits for is not an answer")
check("silent failures are disqualifying",    lambda: meets_bar(_silent)[0] is False,
      "a 90% pass rate with two invisible failures is worse than an 80% one that shouts")
check("an inaccurate one is rejected first",  lambda: "pass rate" in meets_bar(_bad)[1],
      "correctness is checked before anything else")
check("the reason names the failing clause",  lambda: "s/case" in meets_bar(_slow)[1])

## Section 3 &mdash; Candidate, then evidence

The rubric proposes; the measurement disposes. `recommend()` takes the brief and, optionally,
the two measurements from Lab 1.4. With no measurements it returns a candidate and says so.
With them, it may **overrule** the candidate &mdash; and that is the point of the whole lab.

In [ ]:
def recommend(brief: dict, single: dict | None = None, multi: dict | None = None) -> dict:
    """brief: {"name", "deterministic", "fits_one_agent", "separate_authority"}.
    single/multi: {"pass_rate", "silent_failures", "seconds_case"} from a real run.
    """
    candidate = rubric(brief["deterministic"], brief["fits_one_agent"],
                       brief["separate_authority"])
    out = {"name": brief["name"], "candidate": candidate,
           "decision": candidate, "why": "rubric only; no measurement supplied"}
    if not (single and multi):
        return out

    s_ok, s_why = meets_bar(single)
    m_ok, m_why = meets_bar(multi)
    gain = multi["pass_rate"] - single["pass_rate"]

    if candidate in ("supervisor_worker", "peer_to_peer"):
        # the split has to EARN itself against the single agent
        if not m_ok:
            out.update(decision="single_agent", why=f"multi-agent below the bar: {m_why}")
        elif gain < BAR["min_quality_gain"]:
            out.update(decision="single_agent",
                       why=f"split changed the pass rate by only {gain:+.0%}")
        else:
            out.update(decision=candidate, why=f"split earned it: {gain:+.0%} pass rate")
    else:
        out.update(why=f"single arm {s_why}" if s_ok else f"single arm rejected: {s_why}")
    return out

In [ ]:
# --- Self-check: Section 3
SPLIT_BRIEF = {"name": "payment exceptions", "deterministic": False,
               "fits_one_agent": False, "separate_authority": True}

_single_good  = {"pass_rate": 0.80, "silent_failures": 0, "seconds_case": 9.0}
_multi_worth  = {"pass_rate": 0.95, "silent_failures": 0, "seconds_case": 20.0}  # +15%, clean
_multi_silent = {"pass_rate": 0.95, "silent_failures": 1, "seconds_case": 20.0}  # +15%, but hides one
_multi_flat   = {"pass_rate": 0.82, "silent_failures": 0, "seconds_case": 15.0}  # +2%

check("with no measurement it returns the rubric's candidate",
      lambda: recommend(SPLIT_BRIEF)["decision"] == "supervisor_worker")
check("a split that clearly earns it is kept",
      lambda: recommend(SPLIT_BRIEF, _single_good, _multi_worth)["decision"] == "supervisor_worker")
check("a split that hides a failure is overruled despite scoring higher",
      lambda: recommend(SPLIT_BRIEF, _single_good, _multi_silent)["decision"] == "single_agent",
      "a higher pass rate does not buy the right to fail invisibly")
check("the overrule says what was wrong with it",
      lambda: "nothing in the trace" in recommend(SPLIT_BRIEF, _single_good, _multi_silent)["why"])
check("a split that buys nothing is overruled",
      lambda: recommend(SPLIT_BRIEF, _single_good, _multi_flat)["decision"] == "single_agent")
check("the candidate is reported even when overruled",
      lambda: recommend(SPLIT_BRIEF, _single_good, _multi_silent)["candidate"] == "supervisor_worker",
      "a design review needs to see what was proposed AND what the evidence did to it")

## Section 4 &mdash; The design-review table

Four briefs, one table. This is the artifact you take away.

In [ ]:
BRIEFS = [
    {"name": "nightly reconciliation",   "deterministic": True,
     "fits_one_agent": False, "separate_authority": False},
    {"name": "single-desk triage",       "deterministic": False,
     "fits_one_agent": True,  "separate_authority": False},
    {"name": "payment exceptions",       "deterministic": False,
     "fits_one_agent": False, "separate_authority": True},
    {"name": "cross-desk negotiation",   "deterministic": False,
     "fits_one_agent": False, "separate_authority": False},
]

def review_table(briefs, single=None, multi=None) -> str:
    rows = ["  brief                  candidate            decision             why",
            "  " + "-" * 100]
    for b in briefs:
        r = recommend(b, single, multi)
        flag = " " if r["decision"] == r["candidate"] else "!"
        rows.append(f"{flag} {r['name']:22} {r['candidate']:20} {r['decision']:20} {r['why']}")
    return "\n".join(rows)

guard(lambda: print(review_table(BRIEFS)))
print("\n(rows marked ! are where the evidence overruled the rubric)")

In [ ]:
# --- Self-check: Section 4   (built lazily -- a module-level call into a blanked
#     function would crash the cell instead of reporting [TODO])
def _t():
    return review_table(BRIEFS, _single_good, _multi_silent)

check("every brief appears", lambda: all(b["name"] in _t() for b in BRIEFS))
check("the overruled row is flagged", lambda: "!" in _t(),
      "a design review must be able to see where evidence beat the proposal")
check("the deterministic brief is still a workflow", lambda: "workflow" in _t())

## Run it for real

Feed in what you actually got in Lab 1.4, then have the model produce the design-review record.
Notice what it is being asked to do: not to *decide*, but to write up a decision your code
already made and can defend &mdash; and to return it as an object, so it can be filed rather than
re-read.

In [ ]:
class DesignDecision(BaseModel):
    """The record a design review actually needs."""
    architecture: str = Field(description="The architecture chosen, in one or two words")
    rationale: str = Field(description="Two sentences at most, citing only the evidence given")
    revisit_when: str = Field(
        description="One concrete, checkable condition that would reopen this decision")

In [ ]:
if llm_ready():
    def _review():
        # Edit these two dicts with YOUR results from Lab 1.4, then re-run.
        single = {"pass_rate": 1.00, "silent_failures": 0, "seconds_case": 3.4}
        multi  = {"pass_rate": 0.40, "silent_failures": 3, "seconds_case": 4.7}

        print(review_table(BRIEFS, single, multi))
        verdict = recommend(BRIEFS[2], single, multi)

        # The record is an OBJECT, so it can go straight into a design-review system --
        # the same pattern as Lab 1.3's Verdict, applied to your own decision.
        record = get_llm().with_structured_output(DesignDecision).invoke(
            "Produce the design-review record for this decision. Use only the facts given; "
            "do not add claims. `revisit_when` must name a concrete, checkable condition.\n\n"
            f"BRIEF: {verdict['name']}\n"
            f"CANDIDATE FROM RUBRIC: {verdict['candidate']}\n"
            f"DECISION: {verdict['decision']}\n"
            f"EVIDENCE: {verdict['why']}\n"
            f"ACCEPTANCE BAR: {BAR}\n"
            f"MEASURED single={single} multi={multi}")
        print("\n--- the design-review record ---")
        print(f"architecture : {record.architecture}")
        print(f"rationale    : {record.rationale}")
        print(f"revisit when : {record.revisit_when}")
    guard(_review)

### Read it

If the paragraph reads as a justification you would actually sign, the rubric did its job. If it
reads as advocacy for the interesting architecture, look again at which clause let it through.

**What you take from Module 1:** the agent loop, in your own code and in `create_agent`; tools
whose descriptions you can defend with a number; typed contracts between agents, and the bug that
appears the moment you drop one; a rubric that terminates early; and an acceptance bar written
before the result that can overrule your own design preference. Modules 2
and 3 make the agent better. This lab is what stops you building one you did not need.

In [ ]:
score()

## Your turn

1. `rubric()` takes booleans, which assumes someone already made the hard calls. Replace
   `fits_one_agent` with a function of the tool count and argue for the threshold you pick.
2. Add a fifth question &mdash; **can this fail unattended?** &mdash; that can force a supervisor even
   when the rubric would otherwise say single agent. Where in the order does it belong, and why?
3. Take a real brief from your own team, fill in the three booleans honestly, and run it with
   your Lab 1.4 results. If the verdict surprises you, which boolean were you tempted to fill in
   dishonestly?